<a href="https://colab.research.google.com/github/DonChenn/RedditSentimentAnalysis/blob/main/TopicModelLDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

First we import our libraries and We download the Dataset (Liberal vs conservative)

In [2]:
from google.colab import drive
import sys
import kagglehub
import pandas as pd
import os
#!pip install gensim
import gensim
from gensim import corpora
drive.mount('/content/drive')
%run "/content/drive/MyDrive/Colab Notebooks/CleaningPipeline.ipynb"
path = kagglehub.dataset_download("neelgajare/liberals-vs-conservatives-on-reddit-13000-posts")
csv_file = None
for filename in os.listdir(path):
    if filename.endswith(".csv"):
        csv_file = os.path.join(path, filename)
        break

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 57.3 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


100%|██████████| 2.09M/2.09M [00:00<00:00, 2.42MB/s]

Extracting files...


Once dataset is loaded in we replace reddit posts with titles but no text with empty strings. We also create a new column that cobimes the title and the text columns to allow the model to have more information to predict the topic.

In [5]:
if csv_file:
    df = pd.read_csv(csv_file)
    df['Title'] = df['Title'].fillna('')
    df['Text'] = df['Text'].fillna('')
    df['Combined_Content'] = df['Title'] + " " + df['Text']

else:
    print("Error: No CSV file found in the downloaded folder.")

Then we tokenize our combined content columns removing stopwords and non alphabetic chars. Assign an ID to each word. Then We create a bag of words to be able to run the LDA model

In [7]:

processed_data = df['Combined_Content'].apply(clean_tokens).tolist()
dictionary = corpora.Dictionary(processed_data)
corpus = [dictionary.doc2bow(text) for text in processed_data]
lda_model = gensim.models.LdaModel( corpus=corpus, id2word=dictionary, num_topics=10, passes=10, random_state=42)
print("\ntopics")
for idx, topic in lda_model.print_topics(-1):
  print(f"Topic {idx}: {topic}\n")


 Comments topics
Topic 0: 0.017*"woman" + 0.011*"police" + 0.009*"black" + 0.009*"new" + 0.006*"men" + 0.006*"gun" + 0.006*"year" + 0.005*"poll" + 0.005*"court" + 0.005*"violence"

Topic 1: 0.014*"right" + 0.012*"bill" + 0.011*"bidens" + 0.011*"mask" + 0.009*"canadian" + 0.008*"freedom" + 0.008*"act" + 0.008*"protest" + 0.007*"car" + 0.007*"democrat"

Topic 2: 0.031*"ukraine" + 0.027*"russia" + 0.026*"war" + 0.022*"u" + 0.017*"russian" + 0.016*"putin" + 0.015*"biden" + 0.011*"trump" + 0.010*"military" + 0.009*"china"

Topic 3: 0.016*"people" + 0.015*"would" + 0.012*"like" + 0.009*"think" + 0.008*"dont" + 0.008*"im" + 0.008*"one" + 0.008*"get" + 0.006*"make" + 0.006*"know"

Topic 4: 0.028*"land" + 0.023*"wage" + 0.017*"anarchocapitalism" + 0.013*"school" + 0.010*"minimum" + 0.010*"property" + 0.007*"income" + 0.006*"living" + 0.006*"rent" + 0.006*"poor"

Topic 5: 0.011*"state" + 0.010*"worker" + 0.010*"capitalism" + 0.009*"capitalist" + 0.009*"party" + 0.008*"government" + 0.008*"socia